## Setup

In [ ]:
# Install required packages
# !pip install transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Model with Quantization

4-bit quantization lets us fit larger models in memory.

In [ ]:
# Model configuration
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small for demo
# For production: "meta-llama/Llama-2-7b-hf" or "mistralai/Mistral-7B-v0.1"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded: {model_name}")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

## 2. Configure LoRA

Only these parameters will be trained!

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # Rank (higher = more capacity, more params)
    lora_alpha=32,                 # Scaling factor (usually 2x rank)
    target_modules=[               # Which layers to add LoRA to
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,             # Dropout for regularization
    bias="none",                   # Don't train bias
    task_type="CAUSAL_LM"
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Check trainable parameters
model.print_trainable_parameters()
# Should see something like: trainable params: 8M || all params: 1.1B || trainable%: 0.7%

## 3. Prepare Dataset

Let's fine-tune for a specific task: code explanation.

In [ ]:
# Sample dataset - replace with your own!
data = [
    {
        "input": "Explain this code: def factorial(n): return 1 if n <= 1 else n * factorial(n-1)",
        "output": "This is a recursive function that calculates factorial. It returns 1 for base cases (n <= 1), otherwise multiplies n by the factorial of (n-1)."
    },
    {
        "input": "Explain this code: lambda x: x**2",
        "output": "This is a lambda function (anonymous function) that takes a parameter x and returns its square (x**2)."
    },
    # Add more examples...
]

# Or load from Hugging Face
# dataset = load_dataset("your-dataset-name")

# Format as instruction-following
def format_instruction(example):
    """Format data for instruction tuning."""
    return {
        "text": f"""### Instruction:
{example['input']}

### Response:
{example['output']}"""
    }

# For demo, use a small public dataset
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train[:1000]")
print(f"Dataset size: {len(dataset)} examples")
print(f"\nSample:\n{dataset[0]['text'][:200]}...")

## 4. Training Configuration

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lora-finetuned",
    num_train_epochs=1,                   # More epochs for real training
    per_device_train_batch_size=4,        # Adjust based on GPU memory
    gradient_accumulation_steps=4,        # Effective batch = 4 * 4 = 16
    learning_rate=2e-4,                   # Higher than full fine-tuning
    fp16=True,                            # Use mixed precision
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    warmup_steps=50,
    optim="paged_adamw_8bit",             # Memory-efficient optimizer
    report_to="none",                     # or "wandb" for tracking
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

print("Trainer initialized")

## 5. Train!

This will take 5-30 minutes depending on GPU and dataset size.

In [ ]:
# Start training
print("Starting training...")
trainer.train()

# Save LoRA adapter (only ~10-50 MB!)
model.save_pretrained("./lora-adapter")
tokenizer.save_pretrained("./lora-adapter")

print("\nTraining complete!")
print("LoRA adapter saved to ./lora-adapter")

## 6. Test the Fine-tuned Model

In [ ]:
# Load the adapter
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, "./lora-adapter")

def generate_response(prompt, max_length=200):
    """Generate response from fine-tuned model."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
test_prompt = """### Instruction:
Explain what machine learning is in simple terms.

### Response:
"""

print("Before fine-tuning (base model):")
# Would need to load base model separately to compare

print("\nAfter fine-tuning (with LoRA):")
print(generate_response(test_prompt))

## 7. Merge LoRA Adapter (Optional)

Combine adapter with base model for faster inference.

In [ ]:
# Merge adapter into base model
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained("./merged-model")
tokenizer.save_pretrained("./merged-model")

print("Merged model saved!")
print("Now you can load it like a normal model (no PEFT needed)")

## Key Parameters Explained

### LoRA Parameters

**`r` (rank)**: 
- Higher = more capacity but more parameters
- Typical: 8, 16, 32, 64
- Start with 16

**`lora_alpha`**:
- Scaling factor
- Usually 2x rank (e.g., r=16 → alpha=32)

**`target_modules`**:
- Which layers to adapt
- More modules = better but slower
- Query/Key/Value projections are most important

### Training Parameters

**`learning_rate`**:
- LoRA uses higher LR than full fine-tuning
- Typical: 1e-4 to 3e-4
- Start with 2e-4

**`batch_size`**:
- As large as GPU allows
- Use gradient accumulation for effective larger batches

**`epochs`**:
- 1-3 for large datasets
- 5-10 for small datasets
- Watch for overfitting!

## Common Issues & Solutions

### Out of Memory?
```python
# Reduce batch size
per_device_train_batch_size=2

# Increase gradient accumulation
gradient_accumulation_steps=8

# Reduce sequence length
max_seq_length=256

# Use smaller rank
r=8
```

### Model not improving?
```python
# Try higher learning rate
learning_rate=3e-4

# More epochs
num_train_epochs=3

# Higher rank
r=32

# Check your data formatting!
```

### Training too slow?
```python
# Use packing
packing=True

# Reduce target modules
target_modules=["q_proj", "v_proj"]  # Just 2 instead of 4

# Flash Attention (if supported)
attn_implementation="flash_attention_2"
```

## Exercise: Fine-tune for Your Task

1. Prepare your own dataset (minimum 100 examples)
2. Format as instruction-response pairs
3. Fine-tune with LoRA
4. Compare before/after performance

In [ ]:
# Your dataset here
my_data = [
    # {"input": "...", "output": "..."},
    # ...
]

# Convert to Hugging Face dataset format
from datasets import Dataset
my_dataset = Dataset.from_list(my_data)

# Apply formatting
my_dataset = my_dataset.map(format_instruction)

# Train!
# ...

## Key Takeaways

1. **LoRA is efficient**: Train only ~1% of parameters
2. **4-bit quantization**: Fine-tune 7B models on consumer GPUs
3. **Adapter files are tiny**: 10-100 MB vs 10+ GB
4. **Start simple**: r=16, alpha=32, 2e-4 learning rate
5. **Quality data matters**: 100 good examples > 1000 bad ones

## Next Steps

- `04_qlora_efficient.ipynb` - Even more efficient with QLoRA
- `05_instruction_tuning.ipynb` - Better prompts and formats
- `06_evaluation.ipynb` - Measure your fine-tuned model
- `08_deployment.ipynb` - Serve your model in production